# 🔍 SignalIQ — GPU Agent Deployment

Run the SignalIQ Vision Agents backend on a **free Colab T4 GPU** with ngrok tunneling.

**Requirements before running:**
- Stream API Key + Secret
- Google/Gemini API Key
- ElevenLabs API Key
- Deepgram API Key
- ngrok Auth Token (free at [ngrok.com](https://ngrok.com))

> ⚠️ **Runtime → Change runtime type → T4 GPU** before running

## 1️⃣ Check GPU

In [ ]:
!nvidia-smi
import torch
print(f"\nCUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

## 2️⃣ Install Dependencies

In [ ]:
%%capture
# Core Vision Agents SDK with all integrations
!pip install "vision-agents[getstream,gemini,elevenlabs,deepgram,ultralytics]"

# ML libraries for FER
!pip install fer deepface mediapipe opencv-python-headless

# API server
!pip install fastapi uvicorn aiosqlite websockets python-dotenv google-generativeai

# Tunneling
!pip install pyngrok

print("✅ All dependencies installed")

## 3️⃣ API Keys

Enter your API keys below. They are stored only in this session's memory.

In [ ]:
import os
from getpass import getpass

# Collect API keys securely
os.environ["STREAM_API_KEY"] = getpass("Stream API Key: ")
os.environ["STREAM_API_SECRET"] = getpass("Stream API Secret: ")
os.environ["GOOGLE_API_KEY"] = getpass("Google/Gemini API Key: ")
os.environ["ELEVENLABS_API_KEY"] = getpass("ElevenLabs API Key: ")
os.environ["DEEPGRAM_API_KEY"] = getpass("Deepgram API Key: ")

# ngrok for tunneling
os.environ["NGROK_AUTH_TOKEN"] = getpass("ngrok Auth Token: ")

print("\n✅ All API keys set")

## 4️⃣ Upload SignalIQ Agent Code

**Option A:** Clone from GitHub (if pushed)  
**Option B:** Upload the `agent/` folder manually  
**Option C:** Run the cell below to create the code inline

In [ ]:
# Option A: Clone from GitHub (uncomment and replace URL)
# !git clone https://github.com/YOUR_USERNAME/signaliq.git
# %cd signaliq

# Option B: Upload manually via Colab file browser (Files panel on left)
# Upload the entire `agent/` and `api/` directories

# Option C: Mount Google Drive (if code is stored there)
from google.colab import drive
drive.mount('/content/drive')
# Then copy from drive:
# !cp -r /content/drive/MyDrive/signaliq/* /content/

print("Choose your upload method above, then proceed.")

## 5️⃣ Create Config (if not uploaded)

In [ ]:
# Write .env file from the environment variables set above
env_content = f"""STREAM_API_KEY={os.environ.get('STREAM_API_KEY', '')}
STREAM_API_SECRET={os.environ.get('STREAM_API_SECRET', '')}
GOOGLE_API_KEY={os.environ.get('GOOGLE_API_KEY', '')}
ELEVENLABS_API_KEY={os.environ.get('ELEVENLABS_API_KEY', '')}
DEEPGRAM_API_KEY={os.environ.get('DEEPGRAM_API_KEY', '')}
"""

with open('.env', 'w') as f:
    f.write(env_content)

print("✅ .env file created")

## 6️⃣ Verify Agent Code Structure

In [ ]:
import os

required_files = [
    'agent/__init__.py',
    'agent/main.py',
    'agent/config.py',
    'agent/vision/face_detector.py',
    'agent/vision/expression.py',
    'agent/vision/frame_processor.py',
    'agent/intelligence/signal_aggregator.py',
    'agent/intelligence/engagement_scorer.py',
    'agent/intelligence/trigger_logic.py',
    'agent/llm/prompt_templates.py',
    'agent/storage/models.py',
    'agent/storage/session_store.py',
    'api/main.py',
]

print('📁 Checking SignalIQ files...\n')
all_ok = True
for f in required_files:
    exists = os.path.exists(f)
    status = '✅' if exists else '❌ MISSING'
    print(f'  {status}  {f}')
    if not exists:
        all_ok = False

if all_ok:
    print('\n✅ All files present — ready to launch!')
else:
    print('\n❌ Some files are missing. Upload them before proceeding.')

## 7️⃣ Set GPU Device in Config

In [ ]:
# Patch config to use CUDA GPU
from agent.config import config

config.vision.yolo_device = 'cuda'
config.vision.processor_fps = 10

print(f'YOLO device: {config.vision.yolo_device}')
print(f'Processor FPS: {config.vision.processor_fps}')
print(f'Gemini model: {config.llm.gemini_model}')
print(f'Gemini FPS: {config.llm.gemini_fps}')
print('\n✅ Config set for GPU')

## 8️⃣ Test FER Pipeline (Quick Sanity Check)

In [ ]:
import numpy as np
import time

from agent.vision.face_detector import FaceDetector
from agent.vision.expression import ExpressionClassifier
from agent.intelligence.engagement_scorer import EngagementScorer

# Create a dummy frame with a simple synthetic face-like pattern
frame = np.random.randint(100, 200, (480, 640, 3), dtype=np.uint8)

# Test face detector
detector = FaceDetector()
t0 = time.time()
faces = detector.detect(frame)
t1 = time.time()
print(f'Face detection: {len(faces)} faces in {(t1-t0)*1000:.1f}ms')

# Test expression classifier
classifier = ExpressionClassifier()
dummy_face = np.random.randint(100, 200, (64, 64, 3), dtype=np.uint8)
t0 = time.time()
result = classifier.classify(dummy_face)
t1 = time.time()
print(f'Expression: {result.dominant_emotion.value} ({result.confidence:.0%}) in {(t1-t0)*1000:.1f}ms')

# Test engagement scorer
scorer = EngagementScorer()
score = scorer.update(result)
print(f'Engagement score: {score:.1f}/100')
print(f'Trajectory: {scorer.trajectory.value}')

print('\n✅ FER pipeline working on GPU!')

## 9️⃣ Start ngrok Tunnel + API Server

In [ ]:
from pyngrok import ngrok
import subprocess
import threading

# Authenticate ngrok
ngrok.set_auth_token(os.environ['NGROK_AUTH_TOKEN'])

# Start API server in background
def run_api():
    subprocess.run([
        'python', '-m', 'uvicorn', 'api.main:app',
        '--host', '0.0.0.0', '--port', '8000'
    ])

api_thread = threading.Thread(target=run_api, daemon=True)
api_thread.start()

import time
time.sleep(3)  # Wait for server to start

# Create ngrok tunnel for API
api_tunnel = ngrok.connect(8000, 'http')
print(f'\n🌐 API Server (public): {api_tunnel.public_url}')
print(f'   Health check: {api_tunnel.public_url}/health')
print(f'   Sessions API: {api_tunnel.public_url}/api/sessions')
print(f'\n📋 Use this URL in your frontend .env or vite proxy config')

## 🔟 Launch Vision Agents Server

In [ ]:
# Start the Vision Agents server
# This will keep running until you stop the cell
print('🚀 Starting SignalIQ Vision Agent...')
print('   Agent will join Stream calls and process video in real-time')
print('   Press ■ (stop) to shut down\n')

!python agent/main.py serve

## 🧹 Cleanup

In [ ]:
# Kill ngrok tunnels
ngrok.kill()
print('✅ Tunnels closed')

---

## 📖 Architecture on Colab

```
┌─ Your Local Machine ─────────────────────┐
│  React Frontend (localhost:3000)           │
│    ↕ WebSocket + REST                     │
└───────────────────────────────────────────┘
         │  ngrok tunnel
         ▼
┌─ Google Colab (T4 GPU) ───────────────────┐
│  FastAPI Server (:8000)                    │
│  Vision Agents Server (agent/main.py)      │
│    ├── SignalIQProcessor (10fps, CUDA)     │
│    ├── YOLO Pose (CUDA)                    │
│    ├── Gemini Realtime (1fps)              │
│    ├── ElevenLabs TTS                      │
│    └── Deepgram STT                        │
└───────────────────────────────────────────┘
         │  Stream Edge Network
         ▼
┌─ Video Call ──────────────────────────────┐
│  Prospect + Rep on Zoom/Meet/Stream       │
└───────────────────────────────────────────┘
```